# FM crossing analysis — fixed aspect ratio R/L≈½ (3D toric code)

Binder-style crossing analysis of the Fredenhagen–Marcu ratio `O_FM,L(hz)` for
**L = 5, 6, 7** at fixed aspect ratio `R/L ≈ 0.5` (loop `bulkA0.5`: L=6 → R=3;
L=7 → avg of R=3,4 on the same samples; **L=5 → R=2, aspect 0.40** — its bulk can't
host R=3, so it's the least-trusted point). A fixed `R/L` keeps the sizes self-similar,
so `O_FM,L(hz)` curves **cross** at the transition — no `g_c(L)~1/L^x` fit (underdetermined
with 3 sizes); the crossing consistency across pairs IS the systematic error.

Sections: **0** geometry/limits unit test (run first) · **1** load · **2** coarse
(bracket window) · **3** fine (poly fits) · **4** pairwise crossings · **5** bootstrap
errors · **6** final hz_c · **7** figure · **8** diagnostics.


In [ ]:
%matplotlib inline
import glob, json, os
import numpy as np
import matplotlib.pyplot as plt


## Config — the one cell to edit

In [ ]:
# ---- data ----
DATA_DIR = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results/phase_hx0.0_bulkA0.5"
LS       = [5, 6, 7]              # sizes to cross (L=5 is aspect 0.40, a documented caveat)
OBS      = "O"                    # "O" = O_FM

# ---- crossing window + fit ----
WINDOW   = None                   # None -> auto (min-spread region); or (lo, hi) after §2
FIT_DEG  = 1                      # 1 = linear (narrow window); 2 = quadratic robustness check
B_BOOT   = 2000                   # bootstrap resamples for crossing errors

# ---- reference ----
HZ_C_REF = 0.197                  # QMC h_c at hx=0

# ---- diagnostics data (fixed-R extractions; leave as-is, cells skip if absent) ----
DIAG_RDIRS = {r: f"/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results/phase_hx0.0_bulkR{r}"
              for r in (2, 3, 4)}      # #2 uses R=2 across L; #3 uses R=2,3,4 at L=7


## 0 · Geometry + exactly-solvable limits (run FIRST)

Operator-algebra unit test (no ED): the closed loop must commute with every `A_v`
(⟨closed⟩=1) and the open string must anticommute with **exactly 2** `A_v` (its endpoints
→ 2 e-charges → ⟨open⟩=0), giving **O_FM=0 at hz=0 (topological)** and **→1 at hz→∞
(trivial)**. Needs NetKet (uses `Three_TC.fm`); if unavailable it prints the shell command.


In [ ]:
try:
    from Three_TC.model.geometry import ThreeD_ToricCodeGeometry as _Geo
    from Three_TC.fm import verify_fm_geometry, verify_fm_charge_flux, _aspect_sizes
    print(f"{'L':>3} {'R':>6} {'R/L':>5} {'half':>5} {'subset':>6} {'interior':>8} "
          f"{'clo_anti':>8} {'opn_anti':>8} {'O_FM(hz0)':>9} {'O_FM(inf)':>9}")
    ok_all = True
    for L in LS:
        g = _Geo(L, L, L, "OBC")
        for R in _aspect_sizes(g, 2, 0.5)[0]:
            vg = verify_fm_geometry(g, R); vc = verify_fm_charge_flux(g, R)
            ok_all &= vg["ok"] and vc["ok"]
            print(f"{L:>3} {R:>6} {vg['aspect']:>5.2f} {str(vg['half_ok']):>5} "
                  f"{str(vg['open_subset_closed']):>6} {str(vg['vertices_interior']):>8} "
                  f"{vc['closed_anticommuting_Av']:>8} {vc['open_anticommuting_Av']:>8} "
                  f"{str(vc['OFM_hz0_topological']):>9} {str(vc['OFM_hzinf_trivial']):>9}")
    print("\nUNIT TEST:", "PASS" if ok_all else "FAIL  <-- geometry/convention broken")
except Exception as e:
    print("NetKet/Three_TC unavailable here -> run the unit test from the repo venv:")
    print("  .venv/bin/python -c \"from Three_TC.model.geometry import ThreeD_ToricCodeGeometry as G;"
          " from Three_TC.fm import verify_fm_geometry as vg, verify_fm_charge_flux as vc;"
          " [print(L, vg(G(L,L,L,'OBC'),L//2)['ok'], vc(G(L,L,L,'OBC'),L//2)['ok']) for L in (5,6,7)]\"")
    print(f"  (import error: {type(e).__name__}: {e})")


## 1 · Load L=5,6,7 on a common hz grid

In [ ]:
def load_curves(directory, sizes=None):
    recs = [json.load(open(jp)) for jp in sorted(glob.glob(os.path.join(directory, "fm_L*.json")))]
    if not recs:
        raise SystemExit(f"no fm_L*.json in {directory} -- pull the extracted curves first")
    recs = sorted(recs, key=lambda r: r["L"])
    if sizes is not None:
        recs = [r for r in recs if r["L"] in sizes]
    return recs

def get_arrays(rec):
    hz = np.array(rec["field"], float)
    y  = np.array(rec[OBS], float)
    ye = np.array(rec.get("Oe" if OBS == "O" else "mz_e", np.zeros_like(y)), float)
    order = np.argsort(hz)
    return hz[order], y[order], ye[order]

recs = load_curves(DATA_DIR, sizes=set(LS))
curves = {r["L"]: get_arrays(r) for r in recs}
print(f"loaded L = {sorted(curves)}   from {os.path.basename(DATA_DIR)}")
for L, (hz, y, ye) in sorted(curves.items()):
    R = recs[[r['L'] for r in recs].index(L)].get("R")
    asp = recs[[r['L'] for r in recs].index(L)].get("aspect")
    print(f"  L={L}: {len(hz):2d} pts  hz in [{hz.min():.3g},{hz.max():.3g}]  R={R} aspect={asp}"
          + ("" if np.all(np.isfinite(ye) & (ye > 0)) else "   [WARN nan/zero errbars]"))

# common grid (shared hz across all L, within tol) -- crossings only trusted on overlap
tol = 1e-6
common = np.array(sorted(set(np.round(curves[LS[0]][0], 6)).intersection(
    *[set(np.round(curves[L][0], 6)) for L in LS[1:]])))
print(f"\ncommon hz grid ({len(common)} pts): [{common.min():.3g}, {common.max():.3g}]")
for L in LS:
    hz = curves[L][0]
    extra = sorted(set(np.round(hz, 6)) - set(common))
    if extra:
        print(f"  note L={L} has non-common hz {extra} -- crossings restricted to the common range")


## 2 · Coarse pass — overlay curves, bracket the crossing window

In [ ]:
# spread across L on the common grid; the crossing sits near its minimum
def on_grid(L, grid):
    hz, y, ye = curves[L]
    idx = [int(np.argmin(np.abs(hz - g))) for g in grid]
    return y[idx], ye[idx]

Y = np.array([on_grid(L, common)[0] for L in LS])           # (nL, ngrid)
spread = Y.max(0) - Y.min(0)
# crossing = where the WIDEST-L pair difference is smallest, but only in the signal band
# (mean O_FM in (0.05, 0.95)); this ignores the flat O~0 topological tail where spread is
# also ~0 but there is no genuine crossing.
meanO = Y.mean(0)
band = (meanO > 0.05) & (meanO < 0.95)
diff = on_grid(max(LS), common)[0] - on_grid(min(LS), common)[0]
cand = np.where(band)[0]
i_star = int(cand[np.argmin(np.abs(diff[cand]))]) if len(cand) else int(np.argmin(spread))
hz_star = float(common[i_star]); D = float(np.median(np.diff(common)))
auto_window = (round(hz_star - 2 * D, 6), round(hz_star + 2 * D, 6))
print(f"min-spread (visual crossing) near hz={hz_star:.4f}; auto WINDOW = {auto_window}")
print(f"(set WINDOW in the config to override; grid step Delta={D:.4f})")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for L in LS:
    hz, y, ye = curves[L]
    ax[0].errorbar(hz, y, yerr=ye, fmt="o-", ms=4, capsize=2, label=f"L={L}")
ax[0].axvspan(*auto_window, color="gray", alpha=0.15, label="auto window")
ax[0].axvline(HZ_C_REF, ls="--", color="green", lw=1.2, label=f"QMC={HZ_C_REF}")
ax[0].set(xlabel="$h_z$", ylabel=r"$O_{FM}$", title="all L (coarse)"); ax[0].legend()
ax[1].plot(common, spread, "s-"); ax[1].axvline(hz_star, ls="--", color="C3")
ax[1].set(xlabel="$h_z$", ylabel=r"spread $\max_L-\min_L$", title="crossing = min spread")
plt.tight_layout(); plt.show()


## 3 · Fine pass — per-curve polynomial fit in the window

In [ ]:
win = WINDOW if WINDOW is not None else auto_window
lo, hi = win
print(f"window = [{lo}, {hi}]   FIT_DEG = {FIT_DEG}")

def fit_in_window(L, deg):
    hz, y, ye = curves[L]
    m = (hz >= lo - 1e-9) & (hz <= hi + 1e-9)
    if m.sum() < deg + 1:
        raise SystemExit(f"L={L}: only {m.sum()} pts in window, need >= {deg+1}; widen WINDOW")
    return np.polyfit(hz[m], y[m], deg), hz[m], y[m], ye[m]

fig, ax = plt.subplots(figsize=(7, 4.8))
xx = np.linspace(lo, hi, 200)
for L in LS:
    p1, hzm, ym, yem = fit_in_window(L, FIT_DEG)
    ax.errorbar(hzm, ym, yerr=yem, fmt="o", ms=5, capsize=2, label=f"L={L} data")
    ax.plot(xx, np.polyval(p1, xx), "-", label=f"L={L} deg{FIT_DEG}")
    if FIT_DEG == 1:                        # quadratic robustness overlay (if enough pts)
        hz, y, ye = curves[L]; m = (hz >= lo - 1e-9) & (hz <= hi + 1e-9)
        if m.sum() >= 3:
            ax.plot(xx, np.polyval(np.polyfit(hz[m], y[m], 2), xx), ":", lw=1, alpha=0.7)
ax.axvline(HZ_C_REF, ls="--", color="green", lw=1.2, label=f"QMC={HZ_C_REF}")
ax.set(xlabel="$h_z$", ylabel=r"$O_{FM}$", title=f"fine fits in window (dotted = deg2 check)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


## 4 · Pairwise crossings  g*(Li, Lj)

In [ ]:
def poly_cross(h1, y1, h2, y2, window, deg):
    """crossing hz* of two curves, each polyfit(deg) restricted to `window`; nan if none in range."""
    lo, hi = window
    m1 = (h1 >= lo - 1e-9) & (h1 <= hi + 1e-9); m2 = (h2 >= lo - 1e-9) & (h2 <= hi + 1e-9)
    if m1.sum() < deg + 1 or m2.sum() < deg + 1:
        return np.nan
    d = np.polysub(np.polyfit(h1[m1], y1[m1], deg), np.polyfit(h2[m2], y2[m2], deg))
    roots = np.roots(d) if d.size > 1 else np.array([])
    real = sorted(r.real for r in np.atleast_1d(roots)
                  if abs(r.imag) < 1e-9 and lo - 1e-6 <= r.real <= hi + 1e-6)
    return float(real[0]) if real else np.nan

from itertools import combinations
pairs = list(combinations(sorted(LS), 2))
cross = {}
print(f"{'pair':>10} {'g* (deg%d)'%FIT_DEG:>12} {'g* (deg2)':>11}")
for (a, b) in pairs:
    ha, ya, _ = curves[a]; hb, yb, _ = curves[b]
    g1 = poly_cross(ha, ya, hb, yb, win, FIT_DEG)
    g2 = poly_cross(ha, ya, hb, yb, win, 2)
    cross[(a, b)] = g1
    flag = "" if (np.isfinite(g1) and np.isfinite(g2) and abs(g1 - g2) < 0.02) else "  <-- deg1/deg2 differ >0.02: densify grid / recheck window"
    print(f"{str((a,b)):>10} {g1:>12.4f} {g2:>11.4f}{flag}")
print("\n(expected best-conditioned: the widest-L pair (5,7); worst: adjacent (5,6))")


## 5 · Bootstrap the crossings → errors

In [ ]:
rng = np.random.default_rng(0)

def boot_cross(a, b, window, deg, B):
    ha, ya, ea = curves[a]; hb, yb, eb = curves[b]
    if not (np.all(np.isfinite(ea)) and np.all(np.isfinite(eb))):
        return np.array([])                        # need finite MC errors
    out = np.empty(B)
    for k in range(B):
        out[k] = poly_cross(ha, ya + rng.normal(0, ea), hb, yb + rng.normal(0, eb), window, deg)
    return out[np.isfinite(out)]

cross_err = {}
print(f"{'pair':>10} {'g*':>9} {'boot mu':>9} {'sd':>8} {'16-84%':>20} {'frac_ok':>8}")
for (a, b) in pairs:
    bc = boot_cross(a, b, win, FIT_DEG, B_BOOT)
    if bc.size == 0:
        cross_err[(a, b)] = (cross[(a, b)], np.nan); print(f"{str((a,b)):>10}  -- no finite-error bootstrap --"); continue
    sd = float(bc.std()); lo_, hi_ = np.percentile(bc, [16, 84])
    cross_err[(a, b)] = (cross[(a, b)], sd)
    print(f"{str((a,b)):>10} {cross[(a,b)]:>9.4f} {bc.mean():>9.4f} {sd:>8.4f} "
          f"[{lo_:.4f}, {hi_:.4f}] {bc.size/B_BOOT:>8.2f}")


## 6 · Final estimate  hz_c

In [ ]:
vals = np.array([cross[p] for p in pairs], float)
errs = np.array([cross_err[p][1] for p in pairs], float)
finite = np.isfinite(vals) & np.isfinite(errs) & (errs > 0)
# consistency: do all pairwise crossings agree within combined 1-sigma?
consistent = False
if finite.sum() >= 2:
    v, e = vals[finite], errs[finite]
    consistent = bool(np.max(v) - np.min(v) <= np.sqrt(np.max(e)**2 + np.min(e)**2))

if finite.sum() >= 2 and consistent:
    w = 1.0 / errs[finite]**2
    hz_c = float(np.sum(w * vals[finite]) / np.sum(w))
    hz_c_err = float(np.sqrt(1.0 / np.sum(w)))
    method = "weighted mean of consistent crossings"
else:
    g67 = cross.get((6, 7), np.nan); g56 = cross.get((5, 6), np.nan)
    hz_c = float(g67)
    hz_c_err = float(abs(g67 - g56)) if np.isfinite(g67) and np.isfinite(g56) else float("nan")
    method = "g*(6,7) central, |g*(6,7)-g*(5,6)| systematic (crossings drift)"

print("pairwise crossings:")
for p in pairs:
    print(f"   g*{p} = {cross[p]:.4f} +- {cross_err[p][1]:.4f}")
print(f"\nconsistent within errors: {consistent}")
print(f"method: {method}")
print(f"\n==>  hz_c = {hz_c:.4f} +- {hz_c_err:.4f}    (QMC = {HZ_C_REF};  offset = {hz_c - HZ_C_REF:+.4f})")


## 7 · Figure — curves + crossings, zoomed on the window

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
xx = np.linspace(lo, hi, 200)
colors = {L: c for L, c in zip(sorted(LS), plt.cm.viridis(np.linspace(0, 0.8, len(LS))))}
for L in LS:
    hz, y, ye = curves[L]
    m = (hz >= lo - 1e-9) & (hz <= hi + 1e-9)
    ax.errorbar(hz[m], y[m], yerr=ye[m], fmt="o", ms=6, capsize=3, color=colors[L], label=f"L={L}")
    ax.plot(xx, np.polyval(np.polyfit(hz[m], y[m], FIT_DEG), xx), "-", color=colors[L])
for (a, b) in pairs:
    g = cross[(a, b)]
    if np.isfinite(g):
        ax.axvline(g, ls=":", lw=1, color="C3")
        ax.annotate(f"g*{(a,b)}={g:.3f}", (g, ax.get_ylim()[0]), rotation=90,
                    va="bottom", ha="right", fontsize=7, color="C3")
ax.axvline(hz_c, color="k", lw=1.6, label=f"$h_z^c$={hz_c:.4f}")
ax.axvspan(hz_c - hz_c_err, hz_c + hz_c_err, color="k", alpha=0.12)
ax.axvline(HZ_C_REF, ls="--", color="green", lw=1.4, label=f"QMC={HZ_C_REF}")
ax.set(xlabel="$h_z$", ylabel=r"$O_{FM}$ (aspect $R/L\approx0.5$)",
       title=rf"FM crossing analysis, hx=0.0  ->  $h_z^c$={hz_c:.4f}$\pm${hz_c_err:.4f}")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()


## 8 · Diagnostics

**#1 crossing consistency** (from §5/§6): if the three g* agree within errors you're
effectively converged; if they drift monotonically, the spread |g*(6,7)−g*(5,6)| IS the
systematic error. **#2/#3** need the fixed-R extractions (`bulkR2/R3/R4`); those cells
skip gracefully if the dirs are absent. **#4 parity**: is the odd-odd (5,7) crossing
offset from the mixed (5,6)/(6,7)? With 3 sizes this is only a qualitative flag.


In [ ]:
# #1 crossing consistency
print("#1 crossing consistency:")
gs = {p: cross[p] for p in pairs if np.isfinite(cross[p])}
if len(gs) >= 2:
    drift = max(gs.values()) - min(gs.values())
    print(f"   spread of crossings = {drift:.4f}   (systematic-error scale)")
    print(f"   {'consistent within MC errors' if consistent else 'crossings DRIFT -> quote as systematic'}")
# #4 parity
print("\n#4 parity (odd-odd vs mixed):")
if all(np.isfinite(cross[p]) for p in [(5,6),(6,7),(5,7)] if p in cross):
    mixed = np.mean([cross[p] for p in [(5,6),(6,7)] if p in cross])
    print(f"   g*(5,7) [odd-odd] = {cross.get((5,7),np.nan):.4f}  vs  mean mixed (5,6),(6,7) = {mixed:.4f}"
          f"   diff = {cross.get((5,7),np.nan)-mixed:+.4f}")
    print("   (qualitative with 3 sizes; a real parity split needs more L)")


In [ ]:
# #2 fixed-loop convergence: O_FM(R=2; L) vs L at fixed hz -- boundary contamination check
def load_one(directory, L):
    fp = os.path.join(directory, f"fm_L{L}_*.json")
    g = glob.glob(fp)
    return json.load(open(g[0])) if g else None

d2 = DIAG_RDIRS.get(2)
recs2 = [load_one(d2, L) for L in LS] if d2 and os.path.isdir(d2) else []
if recs2 and all(recs2):
    HZ_SHOW = [0.25, 0.275, 0.3]
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for hzc in HZ_SHOW:
        ys = []
        for r in recs2:
            hz = np.array(r["field"]); i = int(np.argmin(np.abs(hz - hzc)))
            ys.append(r["O"][i])
        ax.plot(LS, ys, "o-", label=f"hz={hzc}")
    ax.set(xlabel="$L$", ylabel=r"$O_{FM}(R{=}2;L)$", xticks=LS,
           title="#2 fixed R=2 vs L (flat = box no longer squeezes the loop)")
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print(f"#2 skipped: extract fixed R=2 first ->  R=2 HX=0.0 LS=\"5 6 7\" bash nersc/extract_fm.sh")
    print(f"   then pull into {DIAG_RDIRS.get(2)}")


In [ ]:
# #3 loop-size dependence at L=7: O_FM vs R=2,3,4 at several hz (plateau=deconfined, decay=confined)
L7 = 7
recs7 = {r: load_one(DIAG_RDIRS[r], L7) for r in (2, 3, 4)} if all(
    os.path.isdir(DIAG_RDIRS[r]) for r in (2, 3, 4)) else {}
if recs7 and all(recs7.values()):
    HZ_SHOW = [0.225, 0.25, 0.275, 0.3, 0.325]
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    Rs = [2, 3, 4]
    for hzc in HZ_SHOW:
        ys = []
        for r in Rs:
            hz = np.array(recs7[r]["field"]); i = int(np.argmin(np.abs(hz - hzc)))
            ys.append(recs7[r]["O"][i])
        ax.plot(Rs, ys, "o-", label=f"hz={hzc}")
    ax.set(xlabel=r"loop side $R$ (at L=7)", ylabel=r"$O_{FM}$", xticks=Rs,
           title="#3 R-dependence at L=7 (flatten->deconfined, fall->confined)")
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()
else:
    print("#3 skipped: extract L=7 at R=2,3,4 first ->  for R in 2 3 4: R=$R HX=0.0 LS=\"7\" bash nersc/extract_fm.sh")
    print(f"   then pull into {DIAG_RDIRS}")
